# Train Twin Delayed DDPG (TD3)

Use the public AprendeRL API on `Pendulum-v1`.

$$\tilde a=\operatorname{clip}(\mu_{\bar\theta}(s')+\epsilon),\qquad
y=r+\gamma(1-d)\min_{i=1,2}Q_{\bar\phi_i}(s',\tilde a),\qquad
L_\mu=-\mathbb E[Q_{\phi_1}(s,\mu_\theta(s))].$$

Here $s,a,r,s'$ are observation, action, reward and next observation;
$d$ denotes true termination, $\gamma$ the discount, $Q$ a critic, and
$\theta,\phi$ actor/critic weights. Bars denote target weights; $\mu$ is a
deterministic actor and $\pi$ a stochastic policy. $\alpha$ weights entropy
$\mathcal H$, $\epsilon$ is target noise, and expectations are replay averages.
The episode return measures undiscounted reward; a short run is not a
convergence guarantee.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

from aprenderl import TD3, TD3Config
from aprenderl.utils import evaluate_policy

ENV_ID = "Pendulum-v1"
TOTAL_TIMESTEPS = 20_000
EVALUATION_EPISODES = 3
RENDER_MODE = "human"
SEED = 7
torch.set_num_threads(1)

## Configure and train

Use uniform warmup actions, then the algorithm’s exploration policy. Replay preserves time-limit bootstrapping. Run on CPU with explicit settings; all environments are closed with `try`/`finally`.

In [ ]:
config = TD3Config(
    batch_size=128,
    buffer_size=50_000,
    learning_starts=1_000,
    train_freq=1,
    gradient_steps=1,
    seed=SEED,
)
env = gym.make(ENV_ID)
try:
    agent = TD3(env, config=config, device="cpu")
    agent.learn(total_timesteps=TOTAL_TIMESTEPS)
finally:
    env.close()

## Plot episode returns

The moving average smooths the last ten completed episodes. Compare multiple seeds before drawing conclusions about relative performance.

In [ ]:
returns = np.asarray(agent.episode_returns)
plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.3, label="Episode return")
if len(returns):
    window = min(10, len(returns))
    average = np.convolve(returns, np.ones(window) / window, mode="valid")
    plt.plot(np.arange(window - 1, len(returns)), average, label="Moving mean")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"Training on {ENV_ID}")
plt.legend()
plt.show()

## Watch the deterministic policy

Evaluate with new seeds in a separate rendered environment. The default opens a window; use `RENDER_MODE = "rgb_array"` when running headlessly.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode=RENDER_MODE)
try:
    result = evaluate_policy(
        agent,
        evaluation_env,
        episodes=EVALUATION_EPISODES,
        deterministic=True,
        seed=1_000,
    )
finally:
    evaluation_env.close()
print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")

[Algorithm guide](../docs/algorithms/td3.md) · [From-scratch study notebook](../study/07_continuous_control/03_td3.ipynb)